In [1]:
from transformers import AutoTokenizer,AutoModelForSequenceClassification
import torch
from transformers import BitsAndBytesConfig
import torch.nn.functional as F
MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment"



c:\Users\ACER.MLT\miniconda3\envs\my7_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
quant_config = BitsAndBytesConfig(
    load_in_8bit=True,  # quantize to 8-bit
    llm_int8_threshold=6.0  # optional, helps with some large models
)


In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    quantization_config=quant_config )

In [4]:
tokenizer.vocab_size
LABEL_MAP = {
    0: "Negative",
    1: "Neutral ",
    2: "Positive"
}

In [ ]:
def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        max_length=128,
       
        
    )

    # print(inputs)
    
    with torch.no_grad():
        outputs = model(**inputs)
        probs = F.softmax(outputs.logits, dim=1)

    pred_label_id = torch.argmax(probs).item()
    confidence = probs[0][pred_label_id].item()

    return { 
        "sentiment": LABEL_MAP[pred_label_id],
        "text": text,
        "confidence": round(confidence, 4)
    }



In [6]:
# comments = [
#     "This video is absolutely amazing 🔥🔥",
#     "I don't agree with this explanation",
#     "It was okay, nothing special",
#     "Worst tutorial ever",
#     "Very informative and well explained!",
#     "How do I prevent these other players from flipping over the Monopoly board so I can keep playing?"

# ,"Ray Dalio started hallucinating that he has become a prophet after he became wealthy. His princples are just cringe.",

# "China invested in its infrastructure AND made money for its govt and its own wealthy “oligarchs”. America? Changed its tune with Reagan, got greedy and our infrastructure has deteriorated from a lack of reinvestment and the middle class!"

# ," been trying to find this clip for years now"

# ,"its called greed"

# ," reading this book and it seems like a very useful book. As I always say, learning methods is crucial for executing a way of doing something, but learning principles free you up to adjust those methods/approaches to your own situation and goals with confidence."

# ,"I wish I had Ray as an Uncle or a mentor you can see the care in his attitude"
# ,"this is very funny"
# ]
comments = [
    "I love this product",
    "This is terrible",
    "It is okay, nothing special"
]
p=0
ng=0
n=0
for c in comments:
    x=predict_sentiment(c)
    print(x)
    
    if x['sentiment']=='Positive':
        p+=1
    if x['sentiment']=='Negative':
        ng+=1
    if x['sentiment']=='Neutral ':
        n+=1
    
print(f"positive={p} negative={ng} neutral={n}")

{'sentiment': 'Positive', 'text': 'I love this product', 'confidence': 0.9889}
{'sentiment': 'Negative', 'text': 'This is terrible', 'confidence': 0.9724}
{'sentiment': 'Positive', 'text': 'It is okay, nothing special', 'confidence': 0.7084}
positive=2 negative=1 neutral=0
